In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

# Phase 4 Validation

In [0]:
SELECT
  (SELECT COUNT(*) FROM bronze_efkleia_urbanmart.order_items)                    AS bronze_lines,
  (SELECT COUNT(*) FROM silver_efkleia_urbanmart.etl_rejects
    WHERE source_table = 'order_items')                                          AS rejected_lines,
  (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales)                       AS fact_rows,
  (SELECT SUM(net_amount) FROM silver_efkleia_urbanmart.order_items)             AS silver_net,
  (SELECT SUM(net_amount) FROM gold_efkleia_urbanmart.fact_sales)                AS gold_net;

bronze_lines,rejected_lines,fact_rows,silver_net,gold_net
24111,114,23997,10063191.03,10063191.03


In [0]:
SELECT
  assert_true(
    (SELECT COUNT(*) FROM bronze_efkleia_urbanmart.order_items)
    - (SELECT COUNT(*) FROM silver_efkleia_urbanmart.etl_rejects WHERE source_table = 'order_items')
    = (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales),
    'Reconciliation failed: bronze lines - rejected lines <> fact rows'
  ) AS chk_row_count,
  assert_true(
    (SELECT SUM(net_amount) FROM silver_efkleia_urbanmart.order_items)
    = (SELECT SUM(net_amount) FROM gold_efkleia_urbanmart.fact_sales),
    'Reconciliation failed: silver net_amount <> gold net_amount'
  ) AS chk_net_amount;

chk_row_count,chk_net_amount
null,null


In [0]:
SELECT
  assert_true(
    (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
     LEFT JOIN gold_efkleia_urbanmart.dim_date d ON f.date_key = d.date_key
     WHERE d.date_key IS NULL) = 0,
    'Orphan FK: fact_sales.date_key not found in dim_date'
  ) AS chk_fk_date,
  assert_true(
    (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
     LEFT JOIN gold_efkleia_urbanmart.dim_customer c ON f.customer_key = c.customer_key
     WHERE c.customer_key IS NULL) = 0,
    'Orphan FK: fact_sales.customer_key not found in dim_customer'
  ) AS chk_fk_customer,
  assert_true(
    (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
     LEFT JOIN gold_efkleia_urbanmart.dim_product p ON f.product_key = p.product_key
     WHERE p.product_key IS NULL) = 0,
    'Orphan FK: fact_sales.product_key not found in dim_product'
  ) AS chk_fk_product,
  assert_true(
    (SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
     LEFT JOIN gold_efkleia_urbanmart.dim_store s ON f.store_key = s.store_key
     WHERE s.store_key IS NULL) = 0,
    'Orphan FK: fact_sales.store_key not found in dim_store'
  ) AS chk_fk_store;

chk_fk_date,chk_fk_customer,chk_fk_product,chk_fk_store
null,null,null,null


In [0]:
SELECT
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT order_line_id) FROM gold_efkleia_urbanmart.fact_sales),
              'Duplicate PK in fact_sales (order_line_id)')   AS chk_pk_fact,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT date_key)     FROM gold_efkleia_urbanmart.dim_date),
              'Duplicate PK in dim_date (date_key)')          AS chk_pk_date,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT customer_key) FROM gold_efkleia_urbanmart.dim_customer),
              'Duplicate PK in dim_customer (customer_key)')  AS chk_pk_customer,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT product_key)  FROM gold_efkleia_urbanmart.dim_product),
              'Duplicate PK in dim_product (product_key)')    AS chk_pk_product,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT store_key)    FROM gold_efkleia_urbanmart.dim_store),
              'Duplicate PK in dim_store (store_key)')        AS chk_pk_store;

chk_pk_fact,chk_pk_date,chk_pk_customer,chk_pk_product,chk_pk_store
null,null,null,null,null


In [0]:
SELECT
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT customer_id) FROM gold_efkleia_urbanmart.dim_customer),
              'Duplicate business key in dim_customer (customer_id)') AS chk_bk_customer,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT product_id)  FROM gold_efkleia_urbanmart.dim_product),
              'Duplicate business key in dim_product (product_id)')   AS chk_bk_product,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT store_id)    FROM gold_efkleia_urbanmart.dim_store),
              'Duplicate business key in dim_store (store_id)')       AS chk_bk_store;

chk_bk_customer,chk_bk_product,chk_bk_store
null,null,null


In [0]:
SELECT
  assert_true((SELECT COUNT(*) FROM (
                 SELECT customer_id FROM gold_efkleia_urbanmart.dim_customer_scd2
                 GROUP BY customer_id HAVING COUNT_IF(is_current) <> 1)) = 0,
              'SCD2: a customer does not have exactly one current version')        AS chk_scd2_one_current,
  assert_true((SELECT COUNT(*)
               FROM gold_efkleia_urbanmart.dim_customer_scd2 a
               JOIN gold_efkleia_urbanmart.dim_customer_scd2 b
                 ON  a.customer_id = b.customer_id
                 AND a.customer_scd2_key < b.customer_scd2_key
                 AND a.valid_from <= b.valid_to
                 AND b.valid_from <= a.valid_to) = 0,
              'SCD2: overlapping validity periods for a customer')                  AS chk_scd2_no_overlap,
  assert_true((SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
               LEFT JOIN gold_efkleia_urbanmart.dim_customer_scd2 h
                 ON f.customer_scd2_key = h.customer_scd2_key
               WHERE h.customer_scd2_key IS NULL) = 0,
              'Orphan FK: fact_sales.customer_scd2_key not found in dim_customer_scd2') AS chk_fk_customer_scd2,
  assert_true((SELECT COUNT(*) FROM gold_efkleia_urbanmart.fact_sales f
               JOIN gold_efkleia_urbanmart.dim_customer      c ON f.customer_key      = c.customer_key
               JOIN gold_efkleia_urbanmart.dim_customer_scd2 h ON f.customer_scd2_key = h.customer_scd2_key
               WHERE c.customer_id <> h.customer_id) = 0,
              'SCD1 and SCD2 keys point to different customers')                    AS chk_scd1_scd2_consistent,
  assert_true((SELECT COUNT(*) = COUNT(DISTINCT customer_scd2_key)
               FROM gold_efkleia_urbanmart.dim_customer_scd2),
              'Duplicate PK in dim_customer_scd2')                                 AS chk_pk_customer_scd2;

chk_scd2_one_current,chk_scd2_no_overlap,chk_fk_customer_scd2,chk_scd1_scd2_consistent,chk_pk_customer_scd2
null,null,null,null,null


**Failure test (run once manually, now commented out):** one fact row (L0000001) was deleted on purpose (version 5), leaving 23,996 rows instead of 23,997.
Re-running the reconciliation check then failed with *"Reconciliation failed: bronze lines - rejected lines <> fact rows"*, which proves the checks catch bad data.
The table was restored with `RESTORE TABLE ... TO VERSION AS OF 4` (versions 6 and 7), and all checks passed again.
`DESCRIBE HISTORY` below shows the whole sequence: CREATE → ADD CONSTRAINT ×3 → WRITE → DELETE → RESTORE. Screenshots are in the documentation.

In [0]:
--validation failure test and time travel (DELETE/RESTORE run once manually, now commented out)


-- 1. Note the current version of the fact table
--DESCRIBE HISTORY gold_efkleia_urbanmart.fact_sales;

-- 2. Break the data on purpose: delete one line
--DELETE FROM gold_efkleia_urbanmart.fact_sales WHERE order_line_id = 'L0000001';

-- 3. Re-run Step 2 → chk_row_count must FAIL with your message (screenshot it)

-- 4. Restore the table to the version before the delete (use the number from step 1)
--RESTORE TABLE gold_efkleia_urbanmart.fact_sales TO VERSION AS OF 4;

-- 5. Re-run Step 2 → passes again

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-10-19T16:12:34.000Z,147308696817375,ey26-023@zone01.gr,RESTORE,"Map(timestamp -> null, version -> 4)",null,List(3368108280434528),315fe1f1-02a5-435b-a64f-1b1c79bb31c9,1019-132912-upqe13cv-v2n,6,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 363386)",null,Databricks-Runtime/19.11.x-photon-scala2.13
6,2026-10-19T16:11:27.000Z,147308696817375,ey26-023@zone01.gr,RESTORE,"Map(timestamp -> null, version -> 4)",null,List(3368108280434528),fdcd7cb7-a987-41a5-9650-fc954e344766,1019-132912-upqe13cv-v2n,5,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 363386, numRemovedFiles -> 1, restoredFilesSize -> 363386, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 363386)",null,Databricks-Runtime/19.11.x-photon-scala2.13
5,2026-10-19T16:06:25.000Z,147308696817375,ey26-023@zone01.gr,DELETE,"Map(predicate -> [""(order_line_id#33400 = L0000001)""])",null,List(3368108280434528),db686fa8-c7cc-4415-9b1f-27f3851e2dab,1019-132912-upqe13cv-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1730, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1109, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 619)",null,Databricks-Runtime/19.11.x-photon-scala2.13
4,2026-10-19T15:00:18.000Z,147308696817375,ey26-023@zone01.gr,WRITE,"Map(partitionBy -> [], clusterBy -> [""date_key""], statsOnLoad -> true, mode -> Overwrite, clusteringOnWriteStatus -> fast-path clustering triggered)",null,List(3368108280434528),b2569b3b-86d6-41e3-842b-6efdc798b617,1019-132912-upqe13cv-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 23997, numOutputBytes -> 363386)",null,Databricks-Runtime/19.11.x-photon-scala2.13
3,2026-10-19T15:00:11.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(expr -> net_amount >= 0, name -> ck_fact_net_not_negative)",null,List(3368108280434528),3baec7ee-04bb-4318-8dbe-435231dde133,1019-132912-upqe13cv-v2n,2,WriteSerializable,false,Map(),null,Databricks-Runtime/19.11.x-photon-scala2.13
2,2026-10-19T15:00:09.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(expr -> discount_pct BETWEEN 0 AND 1, name -> ck_fact_discount_range)",null,List(3368108280434528),14eef1a8-bbeb-4556-a417-60034a3610e4,1019-132912-upqe13cv-v2n,1,WriteSerializable,false,Map(),null,Databricks-Runtime/19.11.x-photon-scala2.13
1,2026-10-19T15:00:07.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(expr -> quantity > 0, name -> ck_fact_quantity_positive)",null,List(3368108280434528),efd31373-533c-4610-8ad7-3042f2c1959f,1019-132912-upqe13cv-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/19.11.x-photon-scala2.13
0,2026-10-19T15:00:02.000Z,147308696817375,ey26-023@zone01.gr,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""date_key""], description -> Gold: sales fact, grain = one order line. Includes Completed, Returned and Cancelled orders., isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""b302fd8e-f1b2-4fcb-aa5d-3e4da7088cb0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4728865c-73cf-4792

# Phase 6 · Orchestration (Lakeflow Job)
**Job:** `urbanmart_efkleia_pipeline` · job parameter `catalog = bootcamp` (passed to every notebook through the widget) · serverless compute

00_setup → 01a_bronze → 02_silver → 03_gold → 04_validation

- Each task depends on the previous one: if a task fails, the next ones do not run, so bad data never reaches gold.
- **04_validation is the final gate**: any failing `assert_true` fails the whole job.
- Not in the job: 01b_profiling (read-only exploration and design) and the analytics queries (they read gold and feed the dashboard).
- **Idempotent:** the job ran twice (26/10, 12:01 and 12:15), both succeeded, and time travel shows identical results after both loads (23,997 rows, net 10,063,191.03). See `DESCRIBE HISTORY` below.

In [0]:
-- Time travel: fact_sales after job run 1 vs job run 2
-- SELECT 'v29 - job run 1 (26/10 12:05)' AS load, COUNT(*) AS fact_rows, SUM(net_amount) AS total_net
-- FROM gold_efkleia_urbanmart.fact_sales VERSION AS OF 29
-- UNION ALL
-- SELECT 'v36 - job run 2 (26/10 12:18)', COUNT(*), SUM(net_amount)
-- FROM gold_efkleia_urbanmart.fact_sales VERSION AS OF 36;

load,fact_rows,total_net
v29 - job run 1 (26/10 12:05),23997,10063191.03
v36 - job run 2 (26/10 12:18),23997,10063191.03


**Result:** job run 1 (v29) and job run 2 (v36) contain identical data: 23,997 rows, net 10,063,191.03. The pipeline is idempotent.
`DESCRIBE HISTORY` shows each job-written version tagged with the job name and run ID, and `numOutputRows = 23997` on every WRITE.
Note: older versions (e.g. the 19/10 failure test, v4–v7) remain visible below in `DESCRIBE HISTORY`, but can no longer be queried with time travel: Delta keeps old data files for 7 days by default (`delta.deletedFileRetentionDuration = 168 hours`).

In [0]:
DESCRIBE HISTORY gold_efkleia_urbanmart.fact_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
36,2026-10-26T10:18:35.000Z,147308696817375,ey26-023@zone01.gr,WRITE,"Map(partitionBy -> [], clusterBy -> [""date_key""], statsOnLoad -> true, mode -> Overwrite, clusteringOnWriteStatus -> fast-path clustering triggered)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),3128c6da-fe54-43fe-a69b-0632d5910aea,1026-101528-siakxdft-v2n,35,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 388841, numDeletionVectorsRemoved -> 0, numOutputRows -> 23997, numOutputBytes -> 388841)",null,Databricks-Runtime/19.12.x-photon-scala2.13
35,2026-10-26T10:18:31.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(name -> ck_fact_net_not_negative, expr -> net_amount >= 0)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),98666bfd-3e11-4a07-a1d1-5e26051f2ed9,1026-101528-siakxdft-v2n,34,WriteSerializable,false,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
34,2026-10-26T10:18:30.000Z,147308696817375,ey26-023@zone01.gr,DROP CONSTRAINT,"Map(name -> ck_fact_net_not_negative, expr -> net_amount >= 0, existed -> true)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),49c35e9a-dc23-4480-a116-a2114d420e68,1026-101528-siakxdft-v2n,33,WriteSerializable,true,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
33,2026-10-26T10:18:28.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(name -> ck_fact_discount_range, expr -> discount_pct BETWEEN 0 AND 1)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),24bdda6d-6359-4e48-8c8a-3358fe47c431,1026-101528-siakxdft-v2n,32,WriteSerializable,false,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
32,2026-10-26T10:18:27.000Z,147308696817375,ey26-023@zone01.gr,DROP CONSTRAINT,"Map(name -> ck_fact_discount_range, expr -> discount_pct BETWEEN 0 AND 1, existed -> true)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),ad9a6382-67c4-4785-a8b8-0d9a163485f3,1026-101528-siakxdft-v2n,31,WriteSerializable,true,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
31,2026-10-26T10:18:26.000Z,147308696817375,ey26-023@zone01.gr,ADD CONSTRAINT,"Map(name -> ck_fact_quantity_positive, expr -> quantity > 0)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),8eebc237-382d-4530-8996-1e582bc00bc4,1026-101528-siakxdft-v2n,30,WriteSerializable,false,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
30,2026-10-26T10:18:25.000Z,147308696817375,ey26-023@zone01.gr,DROP CONSTRAINT,"Map(name -> ck_fact_quantity_positive, expr -> quantity > 0, existed -> true)","List(170128914743447, urbanmart_efkleia_pipeline, 202185735015908, 550410405649721, 147308696817375, manual, 7405608981153757)",List(485911799287960),b8e825e6-23a2-415d-9d22-bab7bc061bc0,1026-101528-siakxdft-v2n,29,WriteSerializable,true,Map(),null,Databricks-Runtime/19.12.x-photon-scala2.13
29,2026-10-26T10:05:07.000Z,147308696817375,ey26-023@zone01.gr,WRITE,"Map(partitionBy -> [], clusterBy -> [""date_key""], statsOnLoad -> true, mode -> Overwrite, clusteringOnWriteStatus -> fast-path clustering triggered)","List(170128914743447, urbanmart_efkleia_pipeline, 487546795287209, 872860946835930, 147308696817375, manual, 7405608981153757)",List(485911799287960),5cda94fa-938a-47ca-bc88-00e6a613890a,1026-100154-flcfr8ee-v2n,28,WriteSeri